# Flare and Autoland Diagnostics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Set some plot styling
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (15, 10)

## Load the log file
You can change the `log_file_path` to point to your log file.

In [ ]:
log_file_path = r'f:\Kerbal Space Program\Ships\Script\Integrated Flight Computer\logs\ifc_log_00000127.csv'
try:
    df = pd.read_csv(log_file_path)
    print("Log file loaded successfully.")
except FileNotFoundError:
    print(f"Error: Log file not found at {log_file_path}")

In [ ]:
df.head()

## Data Overview

In [ ]:
print(df.columns.tolist())

## Visualize Flight Profile

In [ ]:
phase_changes = df[df['phase'].diff() != 0]

plt.figure(figsize=(20, 15))

# 1. Altitude vs. Time
plt.subplot(3, 1, 1)
plt.plot(df['t_s'], df['agl_m'], label='Altitude (AGL)')
plt.plot(df['t_s'], df['gs_nom_alt_m'], label='Nominal Glideslope Altitude', linestyle='--')
for i, row in phase_changes.iterrows():
    plt.axvline(x=row['t_s'], color='r', linestyle='--', linewidth=1)
    plt.text(row['t_s'], df['agl_m'].max()*0.9, row['phase'], rotation=90, color='r')
plt.ylabel('Altitude (m)')
plt.title('Altitude and Vertical Speed vs. Time')
plt.legend()
plt.grid(True)

# 2. Vertical Speed vs. Time
plt.subplot(3, 1, 2)
plt.plot(df['t_s'], df['vs_ms'], label='Vertical Speed')
plt.plot(df['t_s'], df['flare_tgt_vs'], label='Flare Target VS', linestyle='--')
for i, row in phase_changes.iterrows():
    plt.axvline(x=row['t_s'], color='r', linestyle='--', linewidth=1)
plt.ylabel('Vertical Speed (m/s)')
plt.xlabel('Time (s)')
plt.legend()
plt.grid(True)

# 3. Airspeed vs. Time
plt.subplot(3, 1, 3)
plt.plot(df['t_s'], df['ias_ms'], label='Indicated Airspeed')
plt.plot(df['t_s'], df['vapp_ms'], label='Approach Speed', linestyle='--')
for i, row in phase_changes.iterrows():
    plt.axvline(x=row['t_s'], color='r', linestyle='--', linewidth=1)
plt.ylabel('Speed (m/s)')
plt.xlabel('Time (s)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Pitch, AoA and Throttle

In [ ]:
plt.figure(figsize=(20, 10))

# 1. Pitch and AoA
plt.subplot(2, 1, 1)
plt.plot(df['t_s'], df['pitch_deg'], label='Pitch')
plt.plot(df['t_s'], df['aoa_deg'], label='Angle of Attack')
plt.plot(df['t_s'], df['flare_fpa_cmd'], label='Flare FPA command', linestyle='--')
for i, row in phase_changes.iterrows():
    plt.axvline(x=row['t_s'], color='r', linestyle='--', linewidth=1)
    plt.text(row['t_s'], df['pitch_deg'].max()*0.9, row['phase'], rotation=90, color='r')
plt.ylabel('Angle (deg)')
plt.title('Pitch, AoA, and Throttle vs. Time')
plt.legend()
plt.grid(True)

# 2. Throttle
plt.subplot(2, 1, 2)
plt.plot(df['t_s'], df['thr_cmd'], label='Throttle Command')
plt.plot(df['t_s'], df['thr_cur'], label='Current Throttle', linestyle='--')
for i, row in phase_changes.iterrows():
    plt.axvline(x=row['t_s'], color='r', linestyle='--', linewidth=1)
plt.ylabel('Throttle')
plt.xlabel('Time (s)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## ILS Tracking Performance

In [ ]:
plt.figure(figsize=(20, 10))

# 1. Localizer
plt.subplot(2, 1, 1)
plt.plot(df['t_s'], df['ils_loc_m'], label='Localizer Deviation (m)')
for i, row in phase_changes.iterrows():
    plt.axvline(x=row['t_s'], color='r', linestyle='--', linewidth=1)
    plt.text(row['t_s'], df['ils_loc_m'].max()*0.9, row['phase'], rotation=90, color='r')
plt.ylabel('Deviation (m)')
plt.title('ILS Tracking vs. Time')
plt.legend()
plt.grid(True)

# 2. Glideslope
plt.subplot(2, 1, 2)
plt.plot(df['t_s'], df['ils_gs_m'], label='Glideslope Deviation (m)')
for i, row in phase_changes.iterrows():
    plt.axvline(x=row['t_s'], color='r', linestyle='--', linewidth=1)
plt.ylabel('Deviation (m)')
plt.xlabel('Time (s)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Flare Details

In [ ]:
flare_df = df[df['phase'] == 'FLARE']
if not flare_df.empty:
    plt.figure(figsize=(20, 15))

    # 1. Flare FPA Command
    plt.subplot(3, 1, 1)
    plt.plot(flare_df['t_s'], flare_df['flare_fpa_cmd'], label='Flare FPA Command')
    plt.ylabel('FPA (deg)')
    plt.title('Flare Details')
    plt.legend()
    plt.grid(True)

    # 2. Flare Target VS
    plt.subplot(3, 1, 2)
    plt.plot(flare_df['t_s'], flare_df['flare_tgt_vs'], label='Flare Target VS')
    plt.plot(flare_df['t_s'], df['vs_ms'], label='Vertical Speed')
    plt.ylabel('Vertical Speed (m/s)')
    plt.legend()
    plt.grid(True)

    # 3. Flare Fraction and Mode
    plt.subplot(3, 1, 3)
    plt.plot(flare_df['t_s'], flare_df['flare_frac'], label='Flare Fraction')
    plt.ylabel('Fraction')
    plt.xlabel('Time (s)')
    plt.legend()
    plt.grid(True)
    
    # Annotate flare mode changes
    flare_mode_changes = flare_df[flare_df['flare_mode'].diff() != 0]
    for i, row in flare_mode_changes.iterrows():
        plt.axvline(x=row['t_s'], color='b', linestyle=':', linewidth=1)
        plt.text(row['t_s'], flare_df['flare_frac'].max()*0.9, row['flare_mode'], rotation=90, color='b')

    plt.tight_layout()
    plt.show()
else:
    print("No flare phase data to plot.")